In [ ]:
# conf = OmegaConf.load('spoken_swag.yaml')

## Validation

In [2]:
# ======================================================================
# CELL 1 — Imports & environment setup
# ======================================================================

import sys
import os
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path.cwd()))

from moshi_dpo_collator import (
    SpokenSwagMoshiCollator,
    MIMI_HOP_LENGTH,
    MIMI_NUM_CODEBOOKS,
    MOSHI_TEXT_PAD_ID,
    MOSHI_TEXT_EPAD_ID,
)
from moshi_dpo_trainer import (
    SLAMMoshiDPOTrainer,
    _apply_delay_pattern,
    _gather_text_logp,
    _gather_audio_logp,
    _build_audio_completion_mask,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

torch.manual_seed(0)
np.random.seed(0)

# Derived constants for reference (Mimi fixed ratios).
SAMPLING_RATE = 24000
FRAME_RATE = 12.5
SAMPLES_PER_FRAME = MIMI_HOP_LENGTH   # = 1920

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"SAMPLING_RATE={SAMPLING_RATE}, FRAME_RATE={FRAME_RATE}, SAMPLES_PER_FRAME={SAMPLES_PER_FRAME}")
print(f"MIMI_NUM_CODEBOOKS={MIMI_NUM_CODEBOOKS}")
print(f"MOSHI_TEXT_PAD_ID={MOSHI_TEXT_PAD_ID}, MOSHI_TEXT_EPAD_ID={MOSHI_TEXT_EPAD_ID}")

Device: cpu
SAMPLING_RATE=24000, FRAME_RATE=12.5, SAMPLES_PER_FRAME=1920
MIMI_NUM_CODEBOOKS=8
MOSHI_TEXT_PAD_ID=3, MOSHI_TEXT_EPAD_ID=0


In [3]:
# ======================================================================
# CELL 2 — Utility: synthetic SpokenSwag-like examples
# ======================================================================

def make_fake_spoken_swag_example(
    prompt_dur: float = 2.5,
    chosen_dur: float = 1.5,
    rejected_dur: float = 1.7,
):
    """Return a dict matching SpokenSwag's row schema with random audio.

    Useful for smoke-testing the pipeline without hitting the real dataset.
    Doesn't include `*_text_token_ids` fields — so only works with
    `use_text_alignment=False` (Option A, audio-only DPO).
    """
    def _random_audio(duration_sec):
        n_samples = int(duration_sec * SAMPLING_RATE)
        return np.random.randn(n_samples).astype(np.float32) * 0.1

    return {
        "prompt":   {"array": _random_audio(prompt_dur),   "sampling_rate": SAMPLING_RATE},
        "chosen":   {"array": _random_audio(chosen_dur),   "sampling_rate": SAMPLING_RATE},
        "rejected": {"array": _random_audio(rejected_dur), "sampling_rate": SAMPLING_RATE},
        "prompt_text":   "the cat sat on the mat",
        "chosen_text":   "then it fell asleep",
        "rejected_text": "then it exploded violently",
    }


def _to_device(batch, device):
    """Move all tensors in a dict to a target device."""
    return {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

In [ ]:
# CELL 3:
# Run test_helpers.py

In [ ]:
# CELL 4 — Load Moshi's tokenizer + feature extractor (HF-integrated path)
# ======================================================================
# IMPORTANT: Moshi has TWO HF repos:
#   - kyutai/moshiko-pytorch-bf16 — raw Kyutai release, no HF config files.
#     Only works with the original `moshi` pip package, NOT with `transformers`.
#   - kmhf/hf-moshiko              — HF-ported repo with config.json,
#     tokenizer_config.json, preprocessor_config.json. This is what we use.
# Both contain the same weights; `kmhf/` is just wrapped for transformers.
# See: https://github.com/huggingface/transformers/issues/41935
#
# There is NO `MoshiProcessor` class — you load the tokenizer and feature
# extractor independently with the usual Auto classes.

from transformers import AutoTokenizer, AutoFeatureExtractor, MoshiConfig

MOSHI_CKPT = "kmhf/hf-moshiko"

tokenizer = AutoTokenizer.from_pretrained(MOSHI_CKPT)
feature_extractor = AutoFeatureExtractor.from_pretrained(MOSHI_CKPT)
cfg = MoshiConfig.from_pretrained(MOSHI_CKPT)

print(f"Tokenizer: {type(tokenizer).__name__}, vocab_size={tokenizer.vocab_size}")
print(f"Feature extractor SR: {feature_extractor.sampling_rate}")
print(f"<pad> → {tokenizer.convert_tokens_to_ids('<pad>')} (expected: {MOSHI_TEXT_PAD_ID})")
print(f"<unk> → {tokenizer.convert_tokens_to_ids('<unk>')} (expected: {MOSHI_TEXT_EPAD_ID})")
print(f"Moshi audio_vocab_size: {cfg.audio_vocab_size}")
print(f"Moshi num_codebooks:    {MIMI_NUM_CODEBOOKS}")

# Sanity checks on the constants baked into the collator.
assert tokenizer.convert_tokens_to_ids("<pad>") == MOSHI_TEXT_PAD_ID, (
    "Text PAD id mismatch — update MOSHI_TEXT_PAD_ID in the collator."
)
assert tokenizer.convert_tokens_to_ids("<unk>") == MOSHI_TEXT_EPAD_ID, (
    "EPAD (repurposed <unk>) id mismatch — update MOSHI_TEXT_EPAD_ID in the collator."
)
assert feature_extractor.sampling_rate == SAMPLING_RATE
print("\n✓ Token ids match expected constants.")

INFO httpx: HTTP Request: HEAD https://huggingface.co/kmhf/hf-moshiko/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: HEAD https://huggingface.co/kmhf/hf-moshiko/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/tokenizer_config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/tokenizer_config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.

Tokenizer class: TokenizersBackend
Vocab size: 32000
pad_token_id: None
pad_token: None
unk_token_id: 0

Feature extractor sampling rate: 24000

special_tokens_map:
  unk_token: '<unk>'

added_tokens_encoder (special and custom):
  '<unk>': 0
  '<s>': 1
  '</s>': 2
  '<pad>': 3

EPAD detection (Kyutai uses token id 0 as padding/silence in codes,
text-stream PAD is id 3 per config.json's existing_text_padding_id):
  '<epad>' → 0
  '[EPAD]' → 0
  '<|epad|>' → 0
  '<pad>' → 3
  '<unk>' → 0


INFO httpx: HTTP Request: HEAD https://huggingface.co/kmhf/hf-moshiko/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/config.json "HTTP/1.1 200 OK"



Moshi config:
  vocab_size: 32000
  audio_vocab_size: 2048
  num_codebooks: 8
  text PAD id (for Inner Monologue stream): 3


In [ ]:
# ======================================================================
# CELL 5 — Load the Moshi model (7B — this is the slow/expensive cell)
# ======================================================================
# After this cell you'll have ~14 GB of VRAM used by moshiko.
# If you're on a small GPU, you can skip to Cell 13 which doesn't need the
# full model, but most of the value of this notebook is in Cells 10-12.
#
# We use `kmhf/hf-moshiko` (HF-integrated variant) — this is what CELL 4 loaded.
# The raw `kyutai/moshiko-pytorch-bf16` repo is NOT loadable via `transformers`
# because it lacks config.json and friends.
from transformers import MoshiForConditionalGeneration

print(f"Loading {MOSHI_CKPT}...")
model = MoshiForConditionalGeneration.from_pretrained(
    MOSHI_CKPT,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
)
model.eval()
print(f"Loaded. Total params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

mimi = model.audio_encoder
print(f"Mimi encoder: {type(mimi).__name__}, device: {next(mimi.parameters()).device}")


Loading kmhf/hf-moshiko...


INFO httpx: HTTP Request: HEAD https://huggingface.co/kmhf/hf-moshiko/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/config.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: HEAD https://huggingface.co/kmhf/hf-moshiko/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
INFO httpx: HTTP Request: HEAD https://huggingface.co/kmhf/hf-moshiko/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/model.safetensors.index.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/kmhf/hf-moshiko/06e3abeb0efb7452a21736d39dcde39411c09982/model.safetensors.index.json "HTTP/1.1 200 OK"
INFO httpx: HTTP Request: GET https://huggingface.co/api/models/kmhf

Loaded. Total params: 7.77B
Audio encoder (Mimi): MimiModel
Mimi device: cuda:0


In [17]:
import importlib
import moshi_dpo_collator
importlib.reload(moshi_dpo_collator)
from moshi_dpo_collator import SpokenSwagMoshiCollator

In [ ]:
# ======================================================================
# CELL 6 — Build a tiny batch and run ONE forward pass
# ======================================================================
# Confirms the collator works end-to-end and shows the shape of
# outputs.audio_logits, which _side_logps relies on.

# Build collator. Teammate's signature: mimi_model required, text ids
# default to MOSHI_TEXT_PAD_ID / MOSHI_TEXT_EPAD_ID so we don't pass them.
collator = SpokenSwagMoshiCollator(
    mimi_model=mimi,
    use_text_alignment=False,
    max_prompt_seconds=3.0,
    max_completion_seconds=2.0,
)

# 1-example batch, ~2.5s total audio.
tiny_features = [make_fake_spoken_swag_example(prompt_dur=1.5, chosen_dur=1.0, rejected_dur=1.0)]
tiny_batch = collator(tiny_features)

print("Tiny batch keys and shapes:")
for k, v in tiny_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)} {v.dtype}")

tiny_batch = _to_device(tiny_batch, DEVICE)

# Run forward pass WITH labels (required to populate audio_logits).
with torch.no_grad():
    outputs = model(
        input_ids=tiny_batch["chosen_input_ids"],
        attention_mask=tiny_batch["chosen_attention_mask"],
        user_audio_codes=tiny_batch["chosen_user_audio_codes"],
        moshi_audio_codes=tiny_batch["chosen_moshi_audio_codes"],
        text_labels=tiny_batch["chosen_input_ids"],             # required
        audio_labels=tiny_batch["chosen_moshi_audio_codes"],    # required
        return_dict=True,
    )

print(f"\nOutput type: {type(outputs).__name__}")
print(f"  logits:       {tuple(outputs.logits.shape)}")
print(f"  audio_logits: {tuple(outputs.audio_logits.shape) if outputs.audio_logits is not None else None}")
print(f"  loss:         {float(outputs.loss):.4f}")
print(f"  depth_loss:   {float(outputs.depth_loss):.4f}")

assert outputs.audio_logits is not None, (
    "audio_logits is None — did both text_labels AND audio_labels get passed?"
)

B, T = tiny_batch["chosen_input_ids"].shape
K = tiny_batch["chosen_moshi_audio_codes"].shape[1]
print(f"\nExpected: B={B}, T={T}, K={K}")
print(f"audio_logits shape {tuple(outputs.audio_logits.shape)} — "
      f"expected (B*T={B*T}, K={K}, V_audio=2048).")

INFO moshi_dpo_collator: Collator init: use_text_alignment=False, pad_id=3, epad_id=0, mimi_encoder=yes


Tiny batch keys and shapes:
  chosen_input_ids: (1, 32) torch.int64
  rejected_input_ids: (1, 32) torch.int64
  chosen_attention_mask: (1, 32) torch.bool
  rejected_attention_mask: (1, 32) torch.bool
  chosen_completion_mask: (1, 32) torch.bool
  rejected_completion_mask: (1, 32) torch.bool
  chosen_moshi_audio_codes: (1, 8, 32) torch.int64
  chosen_user_audio_codes: (1, 8, 32) torch.int64
  rejected_moshi_audio_codes: (1, 8, 32) torch.int64
  rejected_user_audio_codes: (1, 8, 32) torch.int64

Output type: MoshiConditionalGenerationOutputWithPast
Available attributes: ['attentions', 'audio_logits', 'clear', 'copy', 'depth_attentions', 'depth_hidden_states', 'depth_loss', 'depth_past_key_values', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'last_hidden_state', 'logits', 'loss', 'move_to_end', 'past_key_values', 'pop', 'popitem']

logits:       shape=(1, 32, 32000)
audio_logits: shape=None
loss:         None
depth_loss:   None


In [ ]:
# ======================================================================
# CELL 7 — CRITICAL: verify manual batched gather matches HF internal loss
# ======================================================================
# If this passes, our batched log-prob math is correct to float precision.
# If it fails, the diagnostic tells us whether text or audio is off.

# Build trainer stub (bypass TRL's full __init__ which requires more setup
# than we have in a smoke test).
trainer = SLAMMoshiDPOTrainer.__new__(SLAMMoshiDPOTrainer)
trainer.use_text_alignment     = False
trainer.text_loss_weight       = 1.0
trainer.semantic_loss_weight   = 1.0
trainer.acoustic_loss_weight   = 1.0
trainer.audio_bos_token_id     = None  # auto-detect from config

try:
    diag = trainer.verify_manual_loss_matches_hf(model, tiny_batch, tol=5e-3)

    print("✓ Manual batched gather matches HF internal loss.")
    print(f"\nText:")
    print(f"  n_valid positions:  {diag['n_text_valid']}")
    print(f"  HF total CE:        {diag['hf_total_text_CE']:.4f}")
    print(f"  Manual total CE:    {diag['manual_total_text_CE']:.4f}")
    print(f"  Relative diff:      {diag['text_rel_diff']:.2e}")
    print(f"Audio:")
    print(f"  n_valid positions:  {diag['n_audio_valid']}")
    print(f"  HF total CE:        {diag['hf_total_audio_CE']:.4f}")
    print(f"  Manual total CE:    {diag['manual_total_audio_CE']:.4f}")
    print(f"  Relative diff:      {diag['audio_rel_diff']:.2e}")

except AssertionError as e:
    print(f"✗ VERIFICATION FAILED: {e}")
    print()
    print("Next steps:")
    print("  Text mismatch → check _gather_text_logp shift convention.")
    print("  Audio mismatch → check _apply_delay_pattern direction,")
    print("                   audio_logits .view() flatten order, or")
    print("                   the audio completion mask shift.")

except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ======================================================================
# CELL 8 — End-to-end concatenated_forward on a 2-example batch
# ======================================================================

fake_batch = [
    make_fake_spoken_swag_example(prompt_dur=2.5, chosen_dur=1.5, rejected_dur=1.7),
    make_fake_spoken_swag_example(prompt_dur=1.8, chosen_dur=1.2, rejected_dur=1.4),
]

batch_for_dpo = collator(fake_batch)
batch_for_dpo = _to_device(batch_for_dpo, DEVICE)

print("Batch shapes:")
for k, v in batch_for_dpo.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {tuple(v.shape)}")

with torch.no_grad():
    out = trainer.concatenated_forward(model, batch_for_dpo)

print(f"\nconcatenated_forward output:")
print(f"  chosen_logps:    {out['chosen_logps'].cpu().float().tolist()}")
print(f"  rejected_logps:  {out['rejected_logps'].cpu().float().tolist()}")
print(f"  chosen_nll_loss: {float(out['chosen_nll_loss']):.4f}")

B = len(fake_batch)
assert out["chosen_logps"].shape == (B,)
assert out["rejected_logps"].shape == (B,)
assert torch.isfinite(out["chosen_logps"]).all(), "chosen_logps has NaN/inf"
assert torch.isfinite(out["rejected_logps"]).all(), "rejected_logps has NaN/inf"
assert (out["chosen_logps"] < 0).all(), "chosen_logps should all be negative"
assert (out["rejected_logps"] < 0).all(), "rejected_logps should all be negative"
print("\n✓ log-probs are finite, shape [B], and negative.")

In [ ]:
# ======================================================================
# CELL 9 — Gradient check: dummy DPO backward
# ======================================================================
# One step of backward through a tiny subset of params, with a dummy
# reference model (= policy itself, so the loss collapses to -log(0.5)).

model.train()

# Freeze most of the model; enable grad only on the text embedding layer
# (smallest useful surface for a smoke test).
for p in model.parameters():
    p.requires_grad_(False)
for name, p in model.named_parameters():
    if "embed" in name.lower() or "lm_head" in name.lower():
        p.requires_grad_(True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params for smoke test: {trainable/1e6:.1f}M")

# Rebuild the batch WITH grad enabled (the previous run was under no_grad).
batch_grad = collator(tiny_features)
batch_grad = _to_device(batch_grad, DEVICE)

out_grad = trainer.concatenated_forward(model, batch_grad)

# Dummy DPO loss with reference = policy (detached). Expected value: -log(0.5) = 0.6931.
beta = 0.1
ref_chosen   = out_grad["chosen_logps"].detach()
ref_rejected = out_grad["rejected_logps"].detach()

logits_dpo = beta * (
    (out_grad["chosen_logps"] - ref_chosen) -
    (out_grad["rejected_logps"] - ref_rejected)
)
dpo_loss = -F.logsigmoid(logits_dpo).mean()

print(f"\nDPO smoke-test loss: {float(dpo_loss):.6f} (expect ≈ 0.6931)")

dpo_loss.backward()

grad_norm_sq = 0.0
n_with_grad = 0
for p in model.parameters():
    if p.grad is not None:
        assert torch.isfinite(p.grad).all(), "Non-finite gradient!"
        grad_norm_sq += float(p.grad.pow(2).sum())
        n_with_grad += 1

print(f"✓ Backward succeeded. {n_with_grad} params have gradients, "
      f"total grad norm = {grad_norm_sq ** 0.5:.4f}")

model.eval()
for p in model.parameters():
    p.requires_grad_(True)

In [1]:
# ======================================================================
# CELL 10 — Summary
# ======================================================================
print("""
==============================================================================
Validation summary
==============================================================================
✓ Cell 1  — Imports work (collator + trainer).
✓ Cell 4  — Tokenizer has expected PAD=3, EPAD=0 (<unk>) ids.
✓ Cell 5  — Moshi 7B loads; Mimi accessible.
✓ Cell 6  — Collator produces expected shapes; audio_logits populated
            when both text_labels and audio_labels are passed.
✓ Cell 7  — CRITICAL: manual batched gather matches HF internal loss.
            Our DPO log-prob computation is correct.
✓ Cell 8  — concatenated_forward returns finite, negative, [B]-shaped logps.
✓ Cell 9  — DPO backward pass produces finite gradients.

If all passed, the DPO pipeline is verified. Next steps:
  - Wire up TRL's DPOTrainer for real training.
  - For Option B (aligned text stream): write the Whisper-timestamped
    preprocessing script to produce *_text_token_ids fields.

If Cell 7 failed, paste the diagnostic — we fix before doing real training.
==============================================================================
""")


Validation summary
✓ Cell 1  — Imports work (collator + trainer).
✓ Cell 4  — Tokenizer has expected PAD=3, EPAD=0 (<unk>) ids.
✓ Cell 5  — Moshi 7B loads; Mimi accessible.
✓ Cell 6  — Collator produces expected shapes; audio_logits populated
            when both text_labels and audio_labels are passed.
✓ Cell 7  — CRITICAL: manual batched gather matches HF internal loss.
            Our DPO log-prob computation is correct.
✓ Cell 8  — concatenated_forward returns finite, negative, [B]-shaped logps.
✓ Cell 9  — DPO backward pass produces finite gradients.

If all passed, the DPO pipeline is verified. Next steps:
  - Wire up TRL's DPOTrainer for real training.
  - For Option B (aligned text stream): write the Whisper-timestamped
    preprocessing script to produce *_text_token_ids fields.

If Cell 7 failed, paste the diagnostic — we fix before doing real training.

